# TASK #1

## Enhanced Maze Navigation with Multiple Goals

● Description: Modify the given Best-First Search to find a path through a maze with multiple goal points. The algorithm should visit all goal points and return the shortest path covering all goals.

● Challenge: The maze will have several dead ends and multiple goal points at different locations.

In [1]:
import heapq
from collections import defaultdict

def find_path(maze):
    rows = len(maze)
    cols = len(maze[0])
    start = None
    goals = []
    for i in range(rows):
        for j in range(cols):
            if maze[i][j] == 'S':
                start = (i, j)
            elif maze[i][j] == 'G':
                goals.append((i, j))
    if not start or not goals:
        return None
    
    # Directions
    directions = [(-1, 0), (1, 0), (0, -1), (0, 1)]
    
    # Heuristic: distance to nearest remaining goal
    def heuristic(pos, visited):
        remaining = [g for g in goals if g not in visited]
        if not remaining:
            return 0
        return min(abs(pos[0] - g[0]) + abs(pos[1] - g[1]) for g in remaining)
    
    # A* with state (pos, visited_set)
    start_state = (start, frozenset())
    pq = [(0 + heuristic(start, frozenset()), 0, start_state)]  # f, g, state
    came_from = {}
    g_score = defaultdict(lambda: float('inf'))
    g_score[start_state] = 0
    
    while pq:
        f, g, state = heapq.heappop(pq)
        pos, visited = state
        if len(visited) == len(goals):
            # Reconstruct path
            path = []
            current = state
            while current in came_from:
                path.append(current[0])
                current = came_from[current]
            path.append(start)
            path.reverse()
            return path
        
        for di, dj in directions:
            ni, nj = pos[0] + di, pos[1] + dj
            if 0 <= ni < rows and 0 <= nj < cols and maze[ni][nj] != 1:
                new_visited = visited
                if (ni, nj) in goals and (ni, nj) not in visited:
                    new_visited = visited | {(ni, nj)}
                new_state = ((ni, nj), new_visited)
                tentative_g = g + 1
                if tentative_g < g_score[new_state]:
                    came_from[new_state] = state
                    g_score[new_state] = tentative_g
                    f_new = tentative_g + heuristic((ni, nj), new_visited)
                    heapq.heappush(pq, (f_new, tentative_g, new_state))
    return None

# Example usage
maze = [
    ['S', 0, 0, 1],
    [0, 1, 0, 0],
    [0, 0, 0, 'G'],
    [1, 0, 'G', 0]
]
path = find_path(maze)
print("Path:", path)

Path: [(0, 0), (0, 1), (0, 2), (1, 2), (2, 2), (3, 2), (2, 2), (2, 3)]


# TASK #2

## Implement an A* Search where the edge costs change dynamically at random intervals.

The algorithm should adapt to these changes and always find the optimal path. Recompute and adjust paths in real time without restarting the algorithm from scratch.

In [2]:
import heapq
import random
from collections import defaultdict

class DynamicAStar:
    def __init__(self, graph, start, goal):
        self.graph = graph  # dict of dict: graph[node][neighbor] = cost
        self.start = start
        self.goal = goal
        self.g_score = defaultdict(lambda: float('inf'))
        self.g_score[start] = 0
        self.f_score = defaultdict(lambda: float('inf'))
        self.f_score[start] = self.heuristic(start)
        self.came_from = {}
        self.open_set = []
        heapq.heappush(self.open_set, (self.f_score[start], start))
        self.closed_set = set()
    
    def heuristic(self, node):
        # Assume nodes are tuples (x,y), heuristic is Manhattan
        return abs(node[0] - self.goal[0]) + abs(node[1] - self.goal[1])
    
    def update_cost(self, u, v, new_cost):
        # When cost changes, update g_scores and re-open if necessary
        if v in self.g_score and self.g_score[u] + new_cost < self.g_score[v]:
            self.g_score[v] = self.g_score[u] + new_cost
            self.f_score[v] = self.g_score[v] + self.heuristic(v)
            self.came_from[v] = u
            if v not in [item[1] for item in self.open_set]:
                heapq.heappush(self.open_set, (self.f_score[v], v))
            # Also, need to update successors
            for succ in self.graph.get(v, {}):
                if succ in self.closed_set:
                    self.update_cost(v, succ, self.graph[v][succ])
    
    def find_path(self):
        while self.open_set:
            current_f, current = heapq.heappop(self.open_set)
            if current == self.goal:
                return self.reconstruct_path()
            self.closed_set.add(current)
            for neighbor, cost in self.graph.get(current, {}).items():
                tentative_g = self.g_score[current] + cost
                if tentative_g < self.g_score[neighbor]:
                    self.came_from[neighbor] = current
                    self.g_score[neighbor] = tentative_g
                    self.f_score[neighbor] = tentative_g + self.heuristic(neighbor)
                    if neighbor not in self.closed_set:
                        heapq.heappush(self.open_set, (self.f_score[neighbor], neighbor))
        return None
    
    def reconstruct_path(self):
        path = []
        current = self.goal
        while current in self.came_from:
            path.append(current)
            current = self.came_from[current]
        path.append(self.start)
        path.reverse()
        return path

# Example usage
graph = {
    (0,0): {(0,1): 1, (1,0): 1},
    (0,1): {(0,0): 1, (0,2): 1},
    (0,2): {(0,1): 1, (1,2): 1},
    (1,0): {(0,0): 1, (1,1): 1},
    (1,1): {(1,0): 1, (1,2): 1, (2,1): 1},
    (1,2): {(0,2): 1, (1,1): 1},
    (2,1): {(1,1): 1, (2,2): 1},
    (2,2): {(2,1): 1}
}
start = (0,0)
goal = (2,2)
astar = DynamicAStar(graph, start, goal)
path = astar.find_path()
print("Initial path:", path)

# Simulate cost change
astar.update_cost((1,1), (2,1), 10)  # Change cost
# Re-find path
path = astar.find_path()
print("Updated path:", path)

Initial path: [(0, 0), (1, 0), (1, 1), (2, 1), (2, 2)]
Updated path: None


# TASK #3

## Delivery Route Optimization with Time Windows

● Description: Using the Greedy Best-First Search, optimize delivery routes for a set of delivery points. Each delivery point has a specific time window for delivery, and the algorithm must prioritize those with stricter deadlines.

● Challenge: Ensure that the algorithm handles time constraints efficiently while minimizing total travel distance.

In [4]:
def optimize_delivery_route(deliveries, start_pos):
    # deliveries: list of dict {'pos': (x,y), 'deadline': int}
    # Sort by deadline ascending
    deliveries.sort(key=lambda d: d['deadline'])
    visited = set()
    route = []
    current_pos = start_pos
    current_time = 0
    total_distance = 0
    
    for delivery in deliveries:
        pos = delivery['pos']
        if pos in visited:
            continue
        dist = abs(current_pos[0] - pos[0]) + abs(current_pos[1] - pos[1])
        arrival_time = current_time + dist
        if arrival_time <= delivery['deadline']:
            route.append(pos)
            visited.add(pos)
            current_pos = pos
            current_time = arrival_time
            total_distance += dist
        else:
            # Cannot deliver in time, skip or handle
            print(f"Cannot deliver to {delivery['pos']} by deadline {delivery['deadline']}, arrival at {arrival_time}")
    
    return route, total_distance

# Example usage
deliveries = [
    {'pos': (1,1), 'deadline': 5},
    {'pos': (2,2), 'deadline': 10},
    {'pos': (3,3), 'deadline': 15}
]
start = (0,0)
route, distance = optimize_delivery_route(deliveries, start)
print("Route:", route)
print("Total distance:", distance)

Route: [(1, 1), (2, 2), (3, 3)]
Total distance: 6
